In [ ]:
%matplotlib inline

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import vix_tcn_revin_xai_plus as vtrx

CSV_PATH = ROOT / "data" / "raw" / "timeseries_data.csv"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
bundle_path = ROOT / "outputs" / "bundles" / "best_tcn_bundle.pt"
if not bundle_path.exists():
    bundle_path = ROOT / "outputs" / "bundles" / "best_model_bundle.pt"

model, bundle_meta, snapshot = vtrx.load_model_bundle(str(bundle_path), device=DEVICE)
cfg = vtrx.Config(**snapshot["cfg"])
cfg.csv_path = str(CSV_PATH)

target_mode = snapshot.get("target_mode", bundle_meta.get("target_mode", "level"))
print("bundle_path :", bundle_path)
print("target_mode :", target_mode)

In [ ]:
df_raw = vtrx.load_frame(cfg.csv_path, cfg.index_col, list(cfg.drop_cols))

_, _, dl_te, meta = vtrx.build_dataloaders(
    df_raw=df_raw,
    target_col=cfg.target_col,
    seq_len=cfg.seq_len,
    batch_size=cfg.batch_size,
    train_ratio=cfg.train_ratio,
    val_ratio=cfg.val_ratio,
    num_workers=cfg.num_workers,
    pin_memory=(cfg.pin_memory and DEVICE.type == "cuda"),
    persistent_workers=cfg.persistent_workers,
    target_mode=target_mode,
)

In [ ]:
rmse, preds, trues = vtrx.evaluate_level_rmse(
    model=model,
    dl=dl_te,
    device=DEVICE,
    scaler_y=meta["scaler_y"],
    target_mode=meta["target_mode"],
    df_split=meta["df_te"],
    target_col_level=meta["target_col_original"],
    seq_len=cfg.seq_len,
)

dates_te = meta["df_te"].index[cfg.seq_len:cfg.seq_len + len(preds)]

print("test RMSE:", rmse)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dates_te, trues, label="true", lw=1.5)
ax.plot(dates_te, preds, label="pred", lw=1.0, alpha=0.8)
ax.set_title("Prediction vs True")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
errors = np.abs(preds - trues)
sample_ids = np.argsort(errors)[-3:]
sample_ids

In [ ]:
# RevIN 파라미터 확인
if hasattr(model, "revin") and getattr(model.revin, "affine", False):
    w = model.revin.affine_weight.detach().cpu().numpy()
    b = model.revin.affine_bias.detach().cpu().numpy()
    names = meta["feature_names"]

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    axes[0].bar(range(len(w)), w)
    axes[0].axhline(1.0, ls="--", lw=1)
    axes[0].set_title("RevIN affine weights")

    axes[1].bar(range(len(b)), b)
    axes[1].axhline(0.0, ls="--", lw=1)
    axes[1].set_title("RevIN affine bias")
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names, rotation=45, ha="right")

    plt.tight_layout()
    plt.show()
else:
    print("RevIN affine parameters not available.")

In [ ]:
X_scaled_all = vtrx.collect_test_windows(dl_te).cpu()
cam_engine = vtrx.TimeSeriesGradCAMRegression(model, device=DEVICE, aggregate="mean")

In [ ]:
for idx in sample_ids:
    x = X_scaled_all[idx:idx+1].to(DEVICE)
    cam, cam_dict = cam_engine.generate(x)

    window_dates = meta["df_te"].index[idx:idx + cfg.seq_len]

    if cfg.target_col in meta["df_te"].columns:
        target_window = meta["df_te"][cfg.target_col].iloc[idx:idx + cfg.seq_len].values
    elif "log_VIX" in meta["df_te"].columns:
        target_window = np.exp(meta["df_te"]["log_VIX"].iloc[idx:idx + cfg.seq_len].values)
    else:
        target_window = np.zeros(cfg.seq_len)

    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    axes[0].plot(window_dates, target_window, color="black", lw=1.5)
    axes[0].set_title(f"Sample {idx} | target window")

    axes[1].plot(window_dates, cam, color="tab:red", lw=1.5)
    axes[1].fill_between(window_dates, 0, cam, alpha=0.2, color="tab:red")
    axes[1].set_title(f"Sample {idx} | Grad-CAM")
    axes[1].set_ylim(0, max(1e-6, cam.max()) * 1.15)

    plt.tight_layout()
    plt.show()

In [ ]:
cam_engine.remove_hooks()